# Linear Probes

The simplest interpretability tool, and the easiest to over-read. Train a linear
classifier to predict some property from a model's internal activations; if it succeeds,
the property is **linearly decodable** from that layer.

Everything hangs on what that does and does not license you to say. "The information is
present in a linearly accessible form" is well supported. "The model represents this
concept" is a stretch. "The model *uses* this concept" is not supported at all — a probe
is correlational, and no amount of probe accuracy makes it causal.

This notebook builds probes, then builds the controls that stop you fooling yourself.
First topic in the [Interpretability](sparse-autoencoders.ipynb) track;
[Activation Steering](activation-steering.ipynb) is the causal counterpart.

## 1. What & Why

Take activations `h ∈ ℝᵈ` from some layer, take labels `y` for a property you care about
(part of speech, sentiment, truthfulness, whether the answer will be correct), and fit
`ŷ = σ(w·h)`. Probe accuracy is your measurement.

**What makes probing attractive:** it is cheap, it needs no model modification, it works
on any layer, and it can be applied to properties you have labels for but no mechanistic
theory about.

**The two failure modes that make results unreliable:**

1. **The probe learns the task rather than reading it.** A sufficiently expressive probe
   on sufficiently high-dimensional activations can fit almost anything — including random
   labels. High accuracy then reports the *probe's* capacity, not the model's
   representation.
2. **Decodable is not used.** Activations carry far more information than any downstream
   computation consumes. A property can be perfectly decodable and completely ignored by
   the model, and a probe cannot tell the difference.

The controls are: **keep the probe weak** (linear, regularised), **compare against a
control task** with the same label structure but no linguistic content, and **verify
causally** by intervening — which is [steering](activation-steering.ipynb).

## 2. Mental Model

**A metal detector on a beach.**

Sweep it over the sand and it beeps. That tells you metal is present and near enough to
the surface to detect. It does not tell you the beach *uses* the metal for anything, and
it does not tell you whether you found a coin or a bottle cap.

Three consequences that map exactly onto probing practice:

- **A beep is evidence of presence, not of function.** Decodability establishes the
  information is there in a readable form. Whether anything downstream reads it is a
  separate question requiring a separate experiment.
- **Turn the sensitivity up far enough and it beeps everywhere.** A high-capacity probe
  detects structure in noise. That is why probes are kept deliberately weak, and why a
  control task is necessary: it tells you what the detector reads on a beach with no
  metal in it.
- **To prove the metal matters, remove it and see what changes.** That is intervention,
  and it is the only thing that establishes use.

The single most useful discipline: **always run the control.** Probe accuracy alone is
uninterpretable, because you do not know what accuracy an uninformative representation
would have produced.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Probe** | A small classifier trained on frozen activations. Almost always linear, by design. |
| **Linear decodability** | Whether a linear function of the activations recovers the property. The actual claim a probe supports. |
| **Control task** | The same inputs with randomly assigned but consistent labels. Measures probe capacity rather than representation. |
| **Selectivity** | Probe accuracy minus control-task accuracy. The number that should be reported. |
| **Probe capacity** | How expressive the probe is. More capacity means more memorisation and less interpretable results. |
| **Layer sweep** | Probing every layer. Where accuracy peaks is often more informative than its maximum. |
| **Amnesic probing / INLP** | Iteratively removing a direction and measuring the effect on model behaviour — a step toward causality. |
| **Causal vs correlational** | Probes are correlational. [Steering](activation-steering.ipynb) and ablation are causal. |
| **Linear representation hypothesis** | The conjecture that many concepts are encoded as directions. Probing's implicit premise. |
| **Feature leakage** | The probe exploiting a correlate of the label rather than the label's own representation. |

## 4. Setup

NumPy. Activations are synthesised so that ground truth is known — which properties are
encoded, which are decodable-but-unused — because that is the only way to check that a
probing *methodology* reports the truth.

In [1]:
# %pip install numpy

import numpy as np

rng = np.random.default_rng(0)
print("numpy", np.__version__)

numpy 2.5.1


## 5. Worked Examples

### Example 1 — a probe, and the control that makes it meaningful

Build activations in which a concept genuinely is encoded, probe for it, then probe for a
concept that is not there.

In [2]:
D_MODEL, N = 64, 4000

def make_activations(n, seed=0):
    '''Activations carrying two real concepts along fixed directions, plus noise.'''
    r = np.random.default_rng(seed)
    sentiment = r.integers(0, 2, n) * 2 - 1        # +/-1
    tense = r.integers(0, 2, n) * 2 - 1
    dir_sent = r.normal(0, 1, D_MODEL); dir_sent /= np.linalg.norm(dir_sent)
    dir_tense = r.normal(0, 1, D_MODEL); dir_tense /= np.linalg.norm(dir_tense)
    h = (sentiment[:, None] * dir_sent * 1.2
         + tense[:, None] * dir_tense * 1.0
         + r.normal(0, 1, (n, D_MODEL)))
    return h, sentiment, tense, dir_sent

def fit_probe(h, y, steps=800, lr=0.5, l2=1e-3):
    '''A plain regularised linear probe, fitted by gradient ascent on the likelihood.'''
    w = np.zeros(h.shape[1])
    t = (y > 0).astype(float)
    for _ in range(steps):
        p = 1 / (1 + np.exp(-(h @ w)))
        w += lr * (h.T @ (t - p) / len(t) - l2 * w)
    return w

def accuracy(h, y, w):
    return float(np.mean(((h @ w) > 0) == (y > 0)))

h, sentiment, tense, dir_sent = make_activations(N, seed=1)
split = N // 2
h_tr, h_te = h[:split], h[split:]

random_labels = rng.integers(0, 2, N) * 2 - 1        # not encoded anywhere

print(f"{'target':28} {'train acc':>10} {'TEST acc':>10}")
for name, y in [("sentiment (really encoded)", sentiment),
                ("tense (really encoded)", tense),
                ("random labels (control)", random_labels)]:
    w = fit_probe(h_tr, y[:split])
    print(f"{name:28} {accuracy(h_tr, y[:split], w):10.3f} "
          f"{accuracy(h_te, y[split:], w):10.3f}")

print("\nThe two real concepts are recovered well above chance on HELD-OUT data.")
print("Random labels sit at chance on the test set -- exactly as they must, since")
print("nothing in the activations carries them.")
print("\nNote the train/test gap on the random labels: the probe fits the training set")
print("somewhat even though there is nothing to fit. That gap is the whole reason the")
print("next example matters.")

target                        train acc   TEST acc
sentiment (really encoded)        0.895      0.866
tense (really encoded)            0.857      0.826
random labels (control)           0.567      0.507

The two real concepts are recovered well above chance on HELD-OUT data.
Random labels sit at chance on the test set -- exactly as they must, since
nothing in the activations carries them.

Note the train/test gap on the random labels: the probe fits the training set
somewhat even though there is nothing to fit. That gap is the whole reason the
next example matters.


### Example 2 — probe capacity: how to get any answer you like

Increase the probe's expressiveness and watch it "discover" structure that is not there.
This is the most common way probing results mislead.

In [3]:
def fit_mlp_probe(h, y, hidden=256, steps=600, lr=0.05, seed=0):
    '''A deliberately over-powered probe: one hidden layer, no regularisation.'''
    r = np.random.default_rng(seed)
    W1 = r.normal(0, 0.3, (h.shape[1], hidden))
    W2 = r.normal(0, 0.3, hidden)
    t = (y > 0).astype(float)
    for _ in range(steps):
        z = np.tanh(h @ W1)
        p = 1 / (1 + np.exp(-(z @ W2)))
        d = (p - t) / len(t)
        gW2 = z.T @ d
        gz = np.outer(d, W2) * (1 - z ** 2)
        gW1 = h.T @ gz
        W1 -= lr * gW1 * 10
        W2 -= lr * gW2 * 10
    return W1, W2

def mlp_acc(h, y, W1, W2):
    return float(np.mean(((np.tanh(h @ W1) @ W2) > 0) == (y > 0)))

# A SMALL training set, which is the regime real probing studies are usually in.
small = 300
print(f"trained on {small} examples\n")
print(f"{'probe':22} {'target':18} {'train acc':>10} {'TEST acc':>10}")
for pname in ("linear", "MLP (256 hidden)"):
    for tname, y in [("real concept", sentiment), ("RANDOM labels", random_labels)]:
        if pname == "linear":
            w = fit_probe(h[:small], y[:small])
            tr, te = accuracy(h[:small], y[:small], w), accuracy(h_te, y[split:], w)
        else:
            W1, W2 = fit_mlp_probe(h[:small], y[:small])
            tr = mlp_acc(h[:small], y[:small], W1, W2)
            te = mlp_acc(h_te, y[split:], W1, W2)
        print(f"{pname:22} {tname:18} {tr:10.3f} {te:10.3f}")

print("\nThe MLP probe achieves high TRAINING accuracy on RANDOM labels -- there is")
print("nothing there to find, and it finds it anyway. That is memorisation, and if you")
print("report training accuracy you will report a discovery.")
print("\nHeld-out accuracy exposes it. The rule that follows: keep probes linear and")
print("regularised, always report test accuracy, and be suspicious of any probing")
print("result that needed a powerful probe to appear.")

trained on 300 examples

probe                  target              train acc   TEST acc
linear                 real concept            0.957      0.838
linear                 RANDOM labels           0.700      0.521


MLP (256 hidden)       real concept            1.000      0.835


MLP (256 hidden)       RANDOM labels           1.000      0.513

The MLP probe achieves high TRAINING accuracy on RANDOM labels -- there is
nothing there to find, and it finds it anyway. That is memorisation, and if you
report training accuracy you will report a discovery.

Held-out accuracy exposes it. The rule that follows: keep probes linear and
regularised, always report test accuracy, and be suspicious of any probing
result that needed a powerful probe to appear.


### Example 3 — selectivity: the number worth reporting

Hewitt & Liang's control task. Assign each input a random-but-consistent label, probe for
*that*, and subtract. What remains is the accuracy attributable to the representation
rather than to the probe.

In [4]:
def control_labels(h, n_clusters=40, seed=0):
    '''Random but CONSISTENT labels: inputs are bucketed, each bucket gets a fixed
    random label. Structured like the real task, with no representational basis.'''
    r = np.random.default_rng(seed)
    centres = r.normal(0, 1, (n_clusters, h.shape[1]))
    bucket = np.argmin(((h[:, None, :] - centres[None]) ** 2).sum(-1), axis=1)
    lab = r.integers(0, 2, n_clusters) * 2 - 1
    return lab[bucket]

ctrl = control_labels(h, seed=3)

print(f"{'probe':22} {'real acc':>10} {'control acc':>12} {'SELECTIVITY':>13}")
for pname in ("linear", "MLP (256 hidden)"):
    if pname == "linear":
        w_r = fit_probe(h_tr, sentiment[:split]); real = accuracy(h_te, sentiment[split:], w_r)
        w_c = fit_probe(h_tr, ctrl[:split]);      con = accuracy(h_te, ctrl[split:], w_c)
    else:
        A, B = fit_mlp_probe(h_tr, sentiment[:split]); real = mlp_acc(h_te, sentiment[split:], A, B)
        A, B = fit_mlp_probe(h_tr, ctrl[:split]);      con = mlp_acc(h_te, ctrl[split:], A, B)
    print(f"{pname:22} {real:10.3f} {con:12.3f} {real - con:13.3f}")

print("\nSelectivity is the gap between what the probe reads from a real concept and")
print("what it reads from an arbitrary one. A high raw accuracy with LOW selectivity")
print("means the probe was strong enough to fit anything, and your result is about the")
print("probe.")
print("\nReport selectivity, not accuracy. It costs one extra probe fit and it is the")
print("difference between a measurement and an artefact.")

probe                    real acc  control acc   SELECTIVITY
linear                      0.866        0.642         0.224


MLP (256 hidden)            0.847        0.587         0.260

Selectivity is the gap between what the probe reads from a real concept and
what it reads from an arbitrary one. A high raw accuracy with LOW selectivity
means the probe was strong enough to fit anything, and your result is about the
probe.

Report selectivity, not accuracy. It costs one extra probe fit and it is the
difference between a measurement and an artefact.


### Example 4 — decodable is not used

The limitation that matters most, and the one probe accuracy can never detect. Construct
activations where a property is perfectly readable and provably ignored by the output.

In [5]:
n = 4000
r = np.random.default_rng(11)

# Two orthogonal directions in activation space.
u = np.zeros(D_MODEL); u[0] = 1.0        # direction the OUTPUT actually reads
z = np.zeros(D_MODEL); z[1] = 1.0        # direction carrying a decodable-but-unused fact

used_feature = r.integers(0, 2, n) * 2 - 1
unused_feature = r.integers(0, 2, n) * 2 - 1     # independent of the output, by construction

acts = (used_feature[:, None] * u * 1.5
        + unused_feature[:, None] * z * 1.5
        + r.normal(0, 0.5, (n, D_MODEL)))

# The model's output depends ONLY on `used_feature`.
output = (acts @ u > 0).astype(int)

print(f"{'property':34} {'probe test acc':>15} {'affects output?':>17}")
for name, y, truth in [("used_feature (drives the output)", used_feature, "yes"),
                       ("unused_feature (inert)", unused_feature, "NO")]:
    w = fit_probe(acts[:2000], y[:2000])
    print(f"{name:34} {accuracy(acts[2000:], y[2000:], w):15.3f} {truth:>17}")

print("\nBoth are decoded near-perfectly. Only one of them does anything.")
print("\nThe causal test settles it in one line -- ablate the direction and see whether")
print("the output changes:\n")
for name, direction in [("ablate the USED direction", u), ("ablate the UNUSED direction", z)]:
    ablated = acts - np.outer(acts @ direction, direction)
    changed = float(np.mean((ablated @ u > 0).astype(int) != output))
    print(f"  {name:32} outputs changed: {changed:.1%}")

print("\nRemoving the used direction changes the output for a large share of inputs.")
print("Removing the unused one changes nothing at all -- and no probe accuracy could")
print("have told you which was which.")
print("\nThat is the boundary of probing: it establishes that information is PRESENT")
print("and READABLE. To show it is USED you must intervene -- see")
print("[Activation Steering](activation-steering.ipynb).")

property                            probe test acc   affects output?
used_feature (drives the output)             0.998               yes
unused_feature (inert)                       0.998                NO

Both are decoded near-perfectly. Only one of them does anything.

The causal test settles it in one line -- ablate the direction and see whether
the output changes:

  ablate the USED direction        outputs changed: 50.7%
  ablate the UNUSED direction      outputs changed: 0.0%

Removing the used direction changes the output for a large share of inputs.
Removing the unused one changes nothing at all -- and no probe accuracy could
have told you which was which.

That is the boundary of probing: it establishes that information is PRESENT
and READABLE. To show it is USED you must intervene -- see
[Activation Steering](activation-steering.ipynb).


## 6. Gotchas & Pitfalls

- **Reporting training accuracy.** Example 2. Always hold out data, and hold out the same
  way you would for any classifier.
- **No control task.** Example 3. Without it, an accuracy number is uninterpretable.
- **Using a powerful probe.** MLPs and deep probes report their own capacity. If a linear
  probe cannot find it, "the model represents it non-linearly" is one explanation and
  "it is not there" is another; the probe cannot distinguish them.
- **Concluding the model uses the property.** Example 4. Probing is correlational, full
  stop.
- **Label leakage through a correlate.** If your "truthfulness" labels correlate with
  length or topic, the probe may be reading those instead. Balance the probing set.
- **Comparing accuracies across layers of different width.** More dimensions means more
  decodable, mechanically. Normalise or control for it.
- **Probing the residual stream and attributing it to a layer.** The residual stream is
  cumulative; information decodable at layer 20 may have been written at layer 3.
- **Small probing sets.** Probing studies often use a few hundred examples, which is
  precisely where Example 2's memorisation bites.
- **Assuming a direction found by a probe is *the* representation.** Probe weights are one
  of many directions that separate the classes, and typically not the one the model uses —
  which is why difference-in-means often steers better (see
  [Activation Steering](activation-steering.ipynb)).

## 7. When to Use vs Alternatives

| Question | Tool |
|---|---|
| Is this information present and linearly readable? | **Linear probe** with a control task |
| At which layer does it become readable? | **Layer sweep** of probes |
| Does the model actually *use* it? | [**Steering / ablation**](activation-steering.ipynb) — causal |
| What features does this layer represent, without a label set? | [**Sparse autoencoders**](sparse-autoencoders.ipynb) — unsupervised |
| Which components caused this output? | Activation patching / causal tracing |
| Can I remove the property from the representation? | INLP / amnesic probing |

**The honest position.** Probing is the cheapest interpretability method and the weakest
in what it licenses. It answers exactly one question — is this linearly decodable — and
that question is worth answering, especially in layer sweeps where the *pattern* across
depth is informative.

Its limitation is not fixable by better probing: activations carry vastly more information
than the model consumes, so decodability will always over-report. The right role for a
probe is as a **cheap first pass** that tells you where to look, followed by a causal
method that tells you whether it matters. A probing result reported without a control task
and without a causal follow-up should be read as a hypothesis.

## 8. Resources

- [Designing and Interpreting Probes with Control Tasks](https://arxiv.org/abs/1909.03368) — Hewitt & Liang; the selectivity metric of Example 3, and the argument for weak probes.
- [What you can cram into a single vector: Probing sentence embeddings](https://arxiv.org/abs/1805.01070) — the probing methodology that set the pattern.
- [Amnesic Probing: Behavioral Explanation with Amnesic Counterfactuals](https://arxiv.org/abs/2006.00995) — removing a property and measuring behavioural change; the bridge from correlation to causation.
- [Probing Classifiers: Promises, Shortcomings, and Advances](https://arxiv.org/abs/2102.12452) — Belinkov's survey; the clearest statement of what probes do and do not show.
- [The Geometry of Truth](https://arxiv.org/abs/2310.06824) — probing for truthfulness, with causal follow-up done properly.
- [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html) — why linear decodability is subtler than it looks when features are in superposition; leads into [SAEs](sparse-autoencoders.ipynb).